# Task 1 — Inconsistencies and duplicates

**Daniel Ehab** · 58-3825 · MET-02 · CSEN1095 Data Engineering, W26

Student clubs collected sign-ups through an online form for one week, and `club_signups.csv` holds one row per form submission. This notebook cleans it into a table with **one row per student per club** and every value spelled one way, then saves it as `club_signups_clean.csv`.

The notebook follows the Lab 3 rules:

1. **The raw file is never edited.** It is read once into `raw`, and every fix happens on a copy.
2. **Every fix is code.** Each fix is followed by a ✓ check cell that fails loudly if the fix did not work.
3. **Count before and after.** Every step that changes rows says how many.

## Setup

The course's pinned environment from Lab 1: Python 3.12 and pandas 2.3.3. The versions are printed so that anyone who re-runs the notebook can compare.

In [1]:
import sys

import pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)

Python 3.12.14
pandas 2.3.3


## Part A — Inconsistencies

### A.1 Look first

Before fixing anything, we answer the Lab 3 first-look questions for this file: what one row is, whether the key is unique, what type each column has, what the categories look like, and where the nulls are.

The file is read once into `raw`. Nothing assigns to `raw` after this cell; every fix later works on a copy.

In [2]:
raw = pd.read_csv("club_signups.csv")

print(raw.shape)
raw.head()

(39, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


**Types.** Which type did pandas give each column?

In [3]:
raw.dtypes

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

**Distinct values and nulls.** How many different values does each column hold, and is anything missing?

In [4]:
pd.DataFrame({"distinct": raw.nunique(), "missing": raw.isna().sum()})

,distinct,missing
signup_id,36,0
student_id,15,0
full_name,24,0
email,18,0
faculty,13,0
city,9,0
club,13,0
signed_up_at,36,0
fee_paid,9,0


**The key.** The form gave every submission a `signup_id`. Is it unique?

In [5]:
print("rows:", len(raw), "| distinct signup_id:", raw["signup_id"].nunique(), "| unique?", raw["signup_id"].is_unique)

rows: 39 | distinct signup_id: 36 | unique? False


**The categories.** `value_counts()` of every categorical column, including any nulls (`dropna=False`).

In [6]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(raw[col].value_counts(dropna=False), "\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO         

**What we see**, and where each problem gets fixed:

- **One row is one form submission.** 39 rows and 9 columns, as the brief says, and no value is missing anywhere.
- **Every column is text** (`object`). `fee_paid` should be True/False (→ A.4) and `signed_up_at` should be a datetime (→ A.5). The first five rows already show two date shapes: `2026-09-15 13:39` and `15/09/2026 03:12`.
- **The key repeats.** 39 rows but only 36 distinct `signup_id`s, so 3 rows reuse an id that already exists (→ B.1).
- **Too many spellings.** `faculty` has 13 distinct values where 5 are allowed, `club` 13 for 5, and `city` 9 for 3 (→ A.2). The variants come in three kinds: case (`iet`, `CAIRO`), longer or shorter names (`Information Engineering and Technology`, `Pharma`, `Chess Club`, `Alex`), and synonyms (`Soccer`, `Debating`).
- **A value that prints twice.** `Music` appears twice in the `club` counts (5 and 1). Two values that print the same must differ in something invisible (→ checked in A.2).
- **One person, several spellings.** 15 distinct `student_id`s but 24 distinct names and 18 distinct emails, e.g. `AHMED ZAKI` and `MOHAMED ADEL` in capitals (→ A.3).
- **Nine ways to write yes or no** in `fee_paid`: `1`, `yes`, `Y`, `TRUE`, `Yes`, `false`, `N`, `0`, `no` (→ A.4).

### A.2 Faculty, club and city

A.1 showed far more distinct values than the brief allows: 13 faculties for 5, 13 clubs for 5, 9 cities for 3. It also showed `Music` twice in the club counts. We follow the Lab 3 recipe for one spelling per value: look, normalise, map, assert.

**Look closer.** `value_counts()` prints values without quotes, so two values that differ only by a space look identical. Printed as a Python list, each value shows its quotes.

In [7]:
for col in ["faculty", "club", "city"]:
    print(col, sorted(raw[col].unique()), "\n")

faculty ['EMS', 'Engineering and Materials Science', 'IET', 'Information Engineering and Technology', 'MET', 'Management', 'Media Engineering and Technology', 'Pharma', 'Pharmacy', 'ems', 'iet', 'met', 'pharmacy'] 

club ['Chess', 'Chess Club', 'Debate', 'Debating', 'Football', 'Music', 'Music ', 'Robotics', 'Soccer', 'chess', 'debate club', 'football', 'music'] 

city ['Alex', 'Alexandria', 'CAIRO', 'Cairo', 'El Giza', 'Giza', 'alexandria', 'cairo', 'giza'] 



`'Music '` ends with a space: that is the second `Music` from A.1. The other variants differ in case (`iet`, `CAIRO`), in length (`Pharma`, `Chess Club`, `Alex`), or are synonyms (`Soccer`, `Debating`).

**Normalise mechanically.** Stripping spaces and lower-casing fixes the space and every case variant at once. From here on every fix happens on `signups`, a copy of `raw`.

In [8]:
signups = raw.copy()

for col in ["faculty", "club", "city"]:
    before = signups[col].nunique()
    signups[col] = signups[col].str.strip().str.lower()
    print(f"{col}: {before} distinct -> {signups[col].nunique()} after strip + lower")
    print(sorted(signups[col].unique()), "\n")

faculty: 13 distinct -> 9 after strip + lower
['ems', 'engineering and materials science', 'iet', 'information engineering and technology', 'management', 'media engineering and technology', 'met', 'pharma', 'pharmacy'] 

club: 13 distinct -> 9 after strip + lower
['chess', 'chess club', 'debate', 'debate club', 'debating', 'football', 'music', 'robotics', 'soccer'] 

city: 9 distinct -> 5 after strip + lower
['alex', 'alexandria', 'cairo', 'el giza', 'giza'] 



Normalising took `faculty` from 13 to 9 distinct values, `club` from 13 to 9 and `city` from 9 to 5. What is left cannot be fixed mechanically: full names (`information engineering and technology`), short forms (`pharma`, `alex`), a `club` suffix, and synonyms (`soccer`, `debating`).

**Map what is left** with one explicit dictionary per column, with one entry for every spelling printed above. The brief gives two of them: Media Engineering and Technology is MET, and a Soccer club is the Football club. Lower-casing also turned `MET` into `met`, so the canonical spellings need an entry too.

We use `.map()` rather than `.replace()`, so the dictionary is the complete list of accepted spellings: a spelling missing from it becomes empty (`NaN`) instead of staying in the column looking valid. Either way, the check below fails if anything outside the allowed values is left.

In [9]:
FACULTIES = {"MET", "IET", "EMS", "Pharmacy", "Management"}
CLUBS = {"Robotics", "Debate", "Music", "Football", "Chess"}
CITIES = {"Cairo", "Giza", "Alexandria"}

FACULTY_MAP = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
CLUB_MAP = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
CITY_MAP = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

signups["faculty"] = signups["faculty"].map(FACULTY_MAP)
signups["club"] = signups["club"].map(CLUB_MAP)
signups["city"] = signups["city"].map(CITY_MAP)

In [10]:
# ✓ check: only the canonical values remain, and no row was lost
assert len(signups) == len(raw), "a row went missing"
for col, allowed in [("faculty", FACULTIES), ("club", CLUBS), ("city", CITIES)]:
    assert signups[col].isin(allowed).all(), f"{col} still holds {set(signups[col]) - allowed}"
    print(col, signups[col].value_counts().to_dict())
print("✓ faculty, club and city hold only the canonical values")

faculty {'IET': 12, 'Pharmacy': 9, 'EMS': 9, 'MET': 5, 'Management': 4}
club {'Football': 9, 'Chess': 9, 'Music': 8, 'Robotics': 8, 'Debate': 5}
city {'Alexandria': 16, 'Cairo': 16, 'Giza': 7}
✓ faculty, club and city hold only the canonical values


**One more proof.** A student has one faculty and one city, however they typed them. If the dictionaries are right, every `student_id` now shows exactly one of each.

In [11]:
# ✓ check: each student now has a single faculty and a single city
per_student = signups.groupby("student_id")[["faculty", "city"]].nunique()
assert (per_student == 1).all().all(), per_student[(per_student > 1).any(axis=1)]
print(f"✓ all {len(per_student)} students have one faculty and one city")

✓ all 15 students have one faculty and one city


These counts are still per sign-up row, 39 of them, duplicates included. Part B gets the table to one row per student per club.

### A.3 Names and emails

A.1 found 24 distinct names and 18 distinct emails for 15 students, and A.2 showed that this file hides spaces at the edges of values. The brief asks for one spelling per person.

**Look.** How many spellings does each student have? For every student with more than one, the spellings are printed as Python lists, so spaces show.

In [12]:
spellings = signups.groupby("student_id")[["full_name", "email"]].nunique()
messy = spellings[(spellings > 1).any(axis=1)].index
print(f"{len(messy)} of {len(spellings)} students have more than one spelling\n")

for sid in messy:
    rows = signups[signups["student_id"] == sid]
    print(sid, sorted(rows["full_name"].unique()))
    print(" " * len(sid), sorted(rows["email"].unique()), "\n")

7 of 15 students have more than one spelling

55-2992 ['Mohamed Adel', 'mohamed adel']
        ['mohamed.adel@student.guc.edu.eg'] 

55-9245 ['ALI IBRAHIM', 'Ali Ibrahim', 'ali ibrahim']
        ['ali.ibrahim@student.guc.edu.eg'] 

58-8701 ['  Farida  Ibrahim ', 'Farida Ibrahim']
        ['farida.ibrahim@student.guc.edu.eg'] 

61-2495 ['AHMED ADEL', 'ahmed adel']
        ['AHMED.ADEL@STUDENT.GUC.EDU.EG', 'ahmed.adel@student.guc.edu.eg'] 

61-4733 ['  Omar  Adel ', 'OMAR ADEL']
        ['omar.adel@student.guc.edu.eg'] 

61-4844 ['  Mohamed  Adel ', 'MOHAMED ADEL', 'Mohamed Adel']
        [' mohamed.adel@student.guc.edu.eg ', 'MOHAMED.ADEL@STUDENT.GUC.EDU.EG', 'mohamed.adel@student.guc.edu.eg'] 

61-7344 ['  Laila  Mostafa ', 'Laila Mostafa', 'laila mostafa']
        ['LAILA.MOSTAFA@STUDENT.GUC.EDU.EG', 'laila.mostafa@student.guc.edu.eg'] 



7 of the 15 students have more than one spelling. Names differ in case (`ALI IBRAHIM`, `ali ibrahim`) and in spaces, and `'  Farida  Ibrahim '` has spaces at both ends **and** two in the middle. Emails differ in case, and one has a space at each end.

**Fix.** `strip()` only trims the ends, so it cannot remove the double space in the middle. Instead, `split()` breaks a name on any run of spaces and drops the ones at the edges, and `" ".join` puts it back together with single spaces. `title()` then gives everyone the same capitalisation. Emails only need trimming and lower-case.

In [13]:
names_before, emails_before = signups["full_name"].nunique(), signups["email"].nunique()

signups["full_name"] = signups["full_name"].str.split().str.join(" ").str.title()
signups["email"] = signups["email"].str.strip().str.lower()

print(f"distinct names:  {names_before} -> {signups['full_name'].nunique()}")
print(f"distinct emails: {emails_before} -> {signups['email'].nunique()}")

distinct names:  24 -> 14
distinct emails: 18 -> 14


In [14]:
# ✓ check: one spelling per person
assert (signups.groupby("student_id")[["full_name", "email"]].nunique() == 1).all().all(), "a student still has two spellings"
assert not signups["full_name"].str.contains(r"^\s|\s$|\s\s").any(), "a name still has extra spaces"
assert (signups["full_name"] == signups["full_name"].str.title()).all(), "a name is capitalised differently"
assert (signups["email"] == signups["email"].str.strip().str.lower()).all(), "an email is not trimmed and lower-case"
print("✓ every student has one name and one email; no extra spaces; same capitalisation; emails trimmed and lower-case")

✓ every student has one name and one email; no extra spaces; same capitalisation; emails trimmed and lower-case


In [15]:
# the messiest student from the look above, before and after
pd.DataFrame({"before": raw.loc[raw["student_id"] == "61-4844", "full_name"].map(repr),
              "after": signups.loc[signups["student_id"] == "61-4844", "full_name"].map(repr)})

,before,after
2,' Mohamed Adel ','Mohamed Adel'
3,'MOHAMED ADEL','Mohamed Adel'
22,'Mohamed Adel','Mohamed Adel'
23,'Mohamed Adel','Mohamed Adel'


Every student now has one spelling, yet there are 14 distinct names and 14 distinct emails for 15 students.

**Flip the question.** Does each name, and each email, belong to only one `student_id`?

In [16]:
students_per_name = signups.groupby("full_name")["student_id"].nunique()
students_per_email = signups.groupby("email")["student_id"].nunique()
print("names used by more than one student_id: ", students_per_name[students_per_name > 1].to_dict())
print("emails used by more than one student_id:", students_per_email[students_per_email > 1].to_dict())

shared = students_per_name[students_per_name > 1].index
signups.loc[signups["full_name"].isin(shared), ["student_id", "full_name", "email", "faculty", "city"]].drop_duplicates()

names used by more than one student_id:  {'Mohamed Adel': 2}
emails used by more than one student_id: {'mohamed.adel@student.guc.edu.eg': 2}


,student_id,full_name,email,faculty,city
2,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria


**Two people, one name.** `Mohamed Adel` belongs to two `student_id`s: 61-4844 studies EMS and lives in Cairo, 55-2992 studies IET and lives in Alexandria. They even share the email `mohamed.adel@student.guc.edu.eg`. Different ids, faculties and cities: these are two students.

So neither the name nor the email can tell people apart; only `student_id` can. Part B therefore identifies a person by `student_id` alone, and Part C shows with numbers what a name-based key would have done.

### A.4 fee_paid

A.1 showed that `fee_paid` is text (`object`) with nine spellings of yes and no. The brief wants a boolean column: True or False.

**Look.** The spellings after the same mechanical step as in A.2 (strip and lower-case), printed with quotes, and how many rows use each.

In [17]:
fee_spellings = signups["fee_paid"].str.strip().str.lower().value_counts()
print(f"{signups['fee_paid'].nunique()} raw spellings -> {len(fee_spellings)} after strip + lower")
print({repr(k): v for k, v in fee_spellings.items()})

9 raw spellings -> 8 after strip + lower
{"'yes'": 11, "'1'": 7, "'false'": 6, "'y'": 5, "'n'": 4, "'true'": 4, "'0'": 1, "'no'": 1}


Lower-casing merged `Yes` and `yes`, leaving 8 spellings: four that mean yes (`yes`, `1`, `y`, `true`) and four that mean no (`false`, `n`, `0`, `no`).

**The tempting shortcut.** `astype(bool)` looks like the obvious way to get a boolean column. Tried on the column without saving the result:

In [18]:
signups["fee_paid"].astype(bool).value_counts()

fee_paid
True    39
Name: count, dtype: int64

Every row comes out True, including `no`, `false` and `0`: for text, `astype(bool)` only asks whether the string is non-empty. So the meaning has to come from an explicit dictionary.

**Fix.** Map each spelling to True or False. As in A.2, `.map()` turns any spelling missing from the dictionary into `NaN`, which the check below catches. We do not call `astype(bool)` afterwards, because it would quietly turn such a `NaN` into True.

In [19]:
FEE_MAP = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}

signups["fee_paid"] = signups["fee_paid"].str.strip().str.lower().map(FEE_MAP)

In [20]:
# ✓ check: a real boolean column, every spelling known, totals as in the look above
assert signups["fee_paid"].notna().all(), "a fee spelling is missing from FEE_MAP"
assert signups["fee_paid"].dtype == bool, f"fee_paid is {signups['fee_paid'].dtype}"

paid = signups["fee_paid"].value_counts()
yes_spellings = [spelling for spelling, value in FEE_MAP.items() if value]
assert paid[True] == fee_spellings[yes_spellings].sum() and paid.sum() == len(signups)
print(f"✓ fee_paid is {signups['fee_paid'].dtype}: {paid[True]} True, {paid[False]} False")

✓ fee_paid is bool: 27 True, 12 False


`fee_paid` is now a boolean column: 27 True and 12 False, the same split as the spellings in the look. Like the counts in A.2, these are still per sign-up row; Part B will show what changes when resubmissions are removed.